# Using `dzack_research.preamble`

The preamble supplies interactive Sage conveniences, named lattice objects,
recorded fixtures, lattice predicates, involutions, Sterk configurations,
Coxeter diagrams, and the Sage–Julia bridge.

The examples below use canonical objects and leave their mathematical
properties visible.  Run the notebook from top to bottom in the SageMath
kernel.

## Session helpers loaded by `init.sage`

In [ ]:
lmap(lambda x: x + 1, [1, 2, 3])
Σ = sum
Σ([1,2,3])
#!pip install jupyterlab_myst

## Named lattices and constructors for $L_{\mathrm{K3}}$

In [ ]:
for L in [Lattices.U, Lattices.E8, Lattices.E10]:
    show(L)
    print("-"*60)

In [ ]:
show(Lattices.LK3)

In [1]:
A2.<alpha1, alpha2> = Lattices.root_lattice("A", 2)


#alpha1 * alpha1
assert alpha1*alpha1 == -2
assert alpha1*alpha2 == 1
assert alpha1.div() == 1
show(A2)

Integral lattice of rank 2 and signature (0, 2)

In [ ]:
L = Lattices.TEn
show(L)

In [ ]:
L.discriminant_group()

## Predicates and isotropic quotients

In [ ]:
L.discriminant_group().normal_form()

In [ ]:

e, f = Lattices.U.gens()
{
    "E8 is elliptic": Lattices.E8.is_elliptic(),
    "delta(E8)": Lattices.E8.delta(),
    "q(e)": Lattices.U.q(e),
    "e_perp / <e>": e.e_perp_mod_e(),
}

## Recorded fixtures

In [2]:
{name: len(roots) for name, roots in Sterk.sterk_roots().items()}

{'Sterk_1': 12, 'Sterk_2': 10, 'Sterk_3': 12, 'Sterk_4': 11, 'Sterk_5': 14}

## K3-lattice involutions and their eigensublattices

In [ ]:
I_En = Involutions.I_En
L_plus = Lattices.LK3.invariant_lattice(I_En)
L_minus = Lattices.LK3.coinvariant_lattice(I_En)

{
    "I_En squared is the identity": I_En**2 == identity_matrix(ZZ, 22),
    "rank L+": L_plus.rank(),
    "rank L-": L_minus.rank(),
    "signatures": (L_plus.signature_pair(), L_minus.signature_pair()),
}

## Sterk root configurations

In [ ]:
sterk_configurations = Sterk.sterk_roots()
{
    name: {
        "number of roots": len(roots),
        "Gram rank": Lattices.TdP.gram_of(roots).rank(),
    }
    for name, roots in sterk_configurations.items()
}

In [ ]:
isotropic = Sterk.isotropic_vectors()["s4_12"]
Lattices.TdP.b(isotropic, isotropic)

## Coxeter diagrams as Sage parents and their morphisms

In [ ]:
A3_diagram = FiniteCoxeterDiagram.from_cartan_type(["A", 3])
A4_diagram = FiniteCoxeterDiagram.from_cartan_type(["A", 4])
inclusion = A3_diagram.hom([2, 3, 4], codomain=A4_diagram)

{
    "category": A3_diagram.category(),
    "in CoxeterDiagrams": A3_diagram in CoxeterDiagrams(),
    "Coxeter matrix": A3_diagram.coxeter_matrix(),
    "labeled edges": list(A3_diagram.graph().edges(sort=True)),
    "images": inclusion.images(),
}

In [ ]:
all(
    A3_diagram.coxeter_matrix()[s, t]
    == A4_diagram.coxeter_matrix()[
        inclusion(A3_diagram(s)).value,
        inclusion(A3_diagram(t)).value,
    ]
    for s in A3_diagram.index_set()
    for t in A3_diagram.index_set()
)

## Calling Oscar through the Sage–Julia bridge

In [ ]:
JuliaHandle, julia

## Discovering the remaining surfaces

The public modules are intentionally ordinary Python modules, so notebook
completion and `help(...)` expose the rest of each surface:

- `Lattices.namespace()` returns the named lattice namespace;
- `Sterk` exposes the recorded root and isotropic configurations;
- `Involutions` exposes the named K3 involutions.

In [ ]:
named_lattices = tuple(sorted(Lattices.namespace()))
{
    "number of named lattices": len(named_lattices),
    "sample": named_lattices[:12],
    "utilities": (lmap, lzip, to_var_names),
}

In [ ]:
show(Lattices.LK3)

## Free algebras on sets

The free commutative algebra on $\Delta[2]$ should look like a polynomial ring $R[x,y,z]$. The next cells keep the construction at the set level, then inspect the resulting polynomial behavior.

In [ ]:
A = FreeAlgebraOnSet(ZZ, Sets.Δ[2])
a0, a1, a2 = (A.algebra_generator(i) for i in Sets.Δ[2])
(A, a0, a1, a2)

In [ ]:
P.<x, y, z> = PolynomialRing(ZZ, 3)
evaluation = A.Hom(P)(
    lambda monomial: prod(
        (P.gen(i)^e for i, e in monomial.dict().items()),
        P.one(),
    )
)
p = (a0 + a1)^2 * a2
(evaluation(p), (x + y)^2 * z)

A map of generating sets becomes substitution of variables. For $f\colon \Delta[2] \to \Delta[1]$ given by $i \mapsto i \bmod 2$, the induced map sends $a_0,a_1,a_2$ to $b_0,b_1,b_0$.

In [ ]:
S = Sets.Δ[2]
T = Sets.Δ[1]
B = FreeAlgebraOnSet(ZZ, T)
f = SetMorphism(Hom(S, T, Sets()), lambda i: i % 2)
phi = A.induced_hom(f, B)
b0, b1 = (B.algebra_generator(i) for i in T)
(phi(a0), phi(a1), phi(a2), phi((a0 + a1) * a2))

The set map need not be finite. On $\mathbb{N}$, doubling acts on each finite expression without enumerating the whole generating set.

In [ ]:
U = Set(NN)
C = FreeAlgebraOnSet(ZZ, U)
doubling = SetMorphism(Hom(U, U, Sets()), lambda n: 2 * n)
psi = C.induced_hom(doubling, C)
c3, c8 = C.algebra_generator(3), C.algebra_generator(8)
q = 2 * c3 + c8 * c3
(q, psi(q), 2 * C.algebra_generator(6) + C.algebra_generator(16) * C.algebra_generator(6))